In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\mtc\Downloads\employee_productivity.csv")
df

,Employee_ID,Department,Experience_Years,Technical_Skill_Score,Communication_Score,Training_Hours,Projects_Completed,Avg_Weekly_Work_Hours,Productivity_Score,Salary_USD
0,E001,Data Science,5,92,85,120,18,42,88,85000
1,E002,Machine Learning,3,89,78,90,15,45,82,78000
2,E003,Data Analysis,2,75,80,70,10,40,76,62000
3,E004,Data Engineering,6,95,88,150,22,44,91,92000
4,E005,BI & Analytics,4,82,90,110,16,41,85,78000
5,E006,Data Science,1,70,75,60,8,38,70,55000
6,E007,Machine Learning,7,96,84,160,25,46,93,98000
7,E008,Data Engineering,3,85,79,100,14,43,81,76000
8,E009,BI & Analytics,5,87,92,130,19,40,88,83000
9,E010,Data Science,4,90,86,115,17,42,87,87000


In [2]:
# اختيار الأعمدة وإعادة تسميتها
df_new = df[[
    "Experience_Years",
    "Technical_Skill_Score",
    "Communication_Score",
    "Projects_Completed",
    "Avg_Weekly_Work_Hours",
    "Productivity_Score"
]].rename(columns={
    "Experience_Years":      "member_experience_years",
    "Technical_Skill_Score": "technical_skill",
    "Communication_Score":   "communication_skill",
    "Projects_Completed":    "projects_completed",
    "Avg_Weekly_Work_Hours": "workload_hours",
    "Productivity_Score":    "productivity_score"
})
df_new

,member_experience_years,technical_skill,communication_skill,projects_completed,workload_hours,productivity_score
0,5,92,85,18,42,88
1,3,89,78,15,45,82
2,2,75,80,10,40,76
3,6,95,88,22,44,91
4,4,82,90,16,41,85
5,1,70,75,8,38,70
6,7,96,84,25,46,93
7,3,85,79,14,43,81
8,5,87,92,19,40,88
9,4,90,86,17,42,87


In [3]:
# تكبير الداتا + حساب member_on_time_rate
df_big = pd.concat([df_new] * 20, ignore_index=True)
df_big["member_on_time_rate"] = (df_big["productivity_score"] / 100).round(2)
print("df_big shape:", df_big.shape)

df_big shape: (200, 7)


In [4]:
# إنشاء Tasks
n = 2000
np.random.seed(42)

df_tasks = pd.DataFrame({
    "estimated_duration_days": np.random.randint(3, 60, n),
    "progress_percent":        np.random.randint(0, 100, n),
    "priority_level":          np.random.randint(1, 4, n),
    "complexity_level":        np.random.randint(1, 4, n),
    "num_subtasks":            np.random.randint(1, 10, n),
    "status":                  np.random.choice(["ongoing", "completed"], n, p=[0.6, 0.4]),
})

# ربط tasks بالـ users
df_tasks["user_id"] = np.random.choice(df_big.index, n)

# دمج الداتا
df_final = df_tasks.merge(df_big, left_on="user_id", right_index=True)
print("df_final columns:", df_final.columns.tolist())

df_final columns: ['estimated_duration_days', 'progress_percent', 'priority_level', 'complexity_level', 'num_subtasks', 'status', 'user_id', 'member_experience_years', 'technical_skill', 'communication_skill', 'projects_completed', 'workload_hours', 'productivity_score', 'member_on_time_rate']


In [5]:
# Time Features
df_final["days_since_start"] = np.array([
    np.random.randint(1, max(2, x))
    for x in df_final["estimated_duration_days"]
])

df_final["days_remaining"] = (
    df_final["estimated_duration_days"] - df_final["days_since_start"]
)

# Expected Progress & Progress Gap
df_final["expected_progress_percent"] = (
    df_final["days_since_start"] / df_final["estimated_duration_days"] * 100
).round(2)

df_final["progress_gap"] = (
    df_final["expected_progress_percent"] - df_final["progress_percent"]
).round(2)

# User Features
df_final["member_avg_delay_days"] = (
    (1 - (df_final["member_on_time_rate"] / 100)) * 15
).round(1)

df_final["workload_ratio"] = (
    df_final["workload_hours"] / 50
).round(2)

# Task Load Features
df_final["max_allowed_tasks"] = np.random.randint(3, 8, len(df_final))

df_final["member_current_tasks"] = np.array([
    np.random.randint(1, x + 1)
    for x in df_final["max_allowed_tasks"]
])

# Target (Delay) ← آخر خطوة
df_final["delay_label"] = (
    (df_final["days_remaining"] < 0) |
    (df_final["progress_gap"] > 15)
).astype(int)

print("✅ Done!")


✅ Done!


In [13]:
print(df_final["delay_label"].value_counts())


delay_label
0    1237
1     763
Name: count, dtype: int64


In [6]:
# ترتيب الأعمدة النهائية
df_final = df_final[[
    "estimated_duration_days",
    "progress_percent",
    "priority_level",
    "complexity_level",
    "num_subtasks",
    "status",
    "days_since_start",
    "days_remaining",
    "expected_progress_percent",
    "progress_gap",
    "member_experience_years",
    "member_on_time_rate",
    "member_avg_delay_days",
    "max_allowed_tasks",
    "member_current_tasks",
    "workload_ratio",
    "projects_completed",
    "technical_skill",
    "communication_skill",
    "delay_label"
]]

# حفظ الداتا
df_final.to_csv(r"C:\Users\mtc\Downloads\final_dataset.csv", index=False)

print("✅ Shape:", df_final.shape)
print("\n📊 Delay Distribution:")
print(df_final["delay_label"].value_counts())
print(f"\n⚡ Delay %: {df_final['delay_label'].mean()*100:.1f}%")
df_final.head()

✅ Shape: (2000, 20)

📊 Delay Distribution:
delay_label
0    1237
1     763
Name: count, dtype: int64

⚡ Delay %: 38.1%


,estimated_duration_days,progress_percent,priority_level,complexity_level,num_subtasks,status,days_since_start,days_remaining,expected_progress_percent,progress_gap,member_experience_years,member_on_time_rate,member_avg_delay_days,max_allowed_tasks,member_current_tasks,workload_ratio,projects_completed,technical_skill,communication_skill,delay_label
0,41,95,1,2,6,completed,17,24,41.46,-53.54,3,0.82,14.9,3,1,0.90,15,89,78,0
1,54,61,2,3,6,completed,16,38,29.63,-31.37,6,0.91,14.9,7,1,0.88,22,95,88,0
2,31,92,2,1,6,ongoing,8,23,25.81,-66.19,6,0.91,14.9,6,1,0.88,22,95,88,0
3,17,57,1,3,7,completed,7,10,41.18,-15.82,3,0.81,14.9,6,1,0.86,14,85,79,0
4,45,66,3,3,6,ongoing,32,13,71.11,5.11,6,0.91,14.9,6,2,0.88,22,95,88,0


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import pickle

In [17]:
# تنضيف الداتا - تحويل status من نص لرقم
le = LabelEncoder()
df_final['status'] = le.fit_transform(df_final['status'])
print('Nulls:', df_final.isnull().sum().sum())


Nulls: 0


In [19]:
# تقسيم الداتا - 80% training و 20% testing
X = df_final.drop('delay_label', axis=1)
y = df_final['delay_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (1600, 19) | Test: (400, 19)


In [21]:
# بناء الـ Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [23]:
# تقييم الـ Model
y_pred = model.predict(X_test)

print('📊 Model Performance:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred)*100:.1f}%')
print(f'  Precision: {precision_score(y_test, y_pred)*100:.1f}%')
print(f'  Recall:    {recall_score(y_test, y_pred)*100:.1f}%')
print(f'  F1-Score:  {f1_score(y_test, y_pred)*100:.1f}%')
print('\n📋 Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

📊 Model Performance:
  Accuracy:  99.8%
  Precision: 99.4%
  Recall:    100.0%
  F1-Score:  99.7%

📋 Confusion Matrix:
[[235   1]
 [  0 164]]


In [27]:
# حفظ الـ Model
with open('delay_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)


In [29]:
# دالة التنبؤ
def predict_task(task):
    task['status'] = 1 if task['status'] == 'ongoing' else 0
    df_task = pd.DataFrame([task])
    delay_pred = model.predict(df_task)[0]
    delay_prob = model.predict_proba(df_task)[0][1]
    progress   = task['progress_percent']

    print('=' * 40)
    print('📋 Task Prediction Result')
    print('=' * 40)
    if delay_pred == 1:
        print('⚠️  Status        : DELAYED')
    else:
        print('✅  Status        : ON TIME')
    print(f'🎯  Delay Chance  : {delay_prob*100:.1f}%')
    print(f'📈  Progress      : {progress}%')
    print('=' * 40)

print('✅ Function ready!')

✅ Function ready!


In [31]:
# تجربة الـ Model

# Task متأخرة
print('🔴 Task - Delayed:')
predict_task({
    'estimated_duration_days': 10,
    'progress_percent': 20,
    'priority_level': 3,
    'complexity_level': 3,
    'num_subtasks': 8,
    'status': 'ongoing',
    'days_since_start': 8,
    'days_remaining': 2,
    'expected_progress_percent': 80.0,
    'progress_gap': 60.0,
    'member_experience_years': 1,
    'member_on_time_rate': 0.5,
    'member_avg_delay_days': 7.5,
    'max_allowed_tasks': 7,
    'member_current_tasks': 7,
    'workload_ratio': 0.9,
    'projects_completed': 2,
    'technical_skill': 55,
    'communication_skill': 60
})

# Task في الموعد
print('\n🟢 Task - On Time:')
predict_task({
    'estimated_duration_days': 14,
    'progress_percent': 75,
    'priority_level': 1,
    'complexity_level': 1,
    'num_subtasks': 2,
    'status': 'ongoing',
    'days_since_start': 9,
    'days_remaining': 5,
    'expected_progress_percent': 64.3,
    'progress_gap': -10.7,
    'member_experience_years': 7,
    'member_on_time_rate': 0.95,
    'member_avg_delay_days': 0.75,
    'max_allowed_tasks': 5,
    'member_current_tasks': 2,
    'workload_ratio': 0.4,
    'projects_completed': 20,
    'technical_skill': 90,
    'communication_skill': 88
})

🔴 Task - Delayed:
📋 Task Prediction Result
⚠️  Status        : DELAYED
🎯  Delay Chance  : 99.0%
📈  Progress      : 20%

🟢 Task - On Time:
📋 Task Prediction Result
✅  Status        : ON TIME
🎯  Delay Chance  : 1.0%
📈  Progress      : 75%
